In [2]:
!pip install datasets transformers evaluate seqeval torch

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

# --- TASK 1: Dataset Selection ---
dataset_name = "eriktks/conll2003"
print(f"Loading Dataset: {dataset_name}\n")
dataset = load_dataset(dataset_name, revision="convert/parquet")

pos_labels = dataset['train'].features['pos_tags'].feature.names
chunk_labels = dataset['train'].features['chunk_tags'].feature.names

# --- TASK 2: Preprocessing & Alignment ---
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("Data preprocessing and label alignment complete!")

Loading Dataset: eriktks/conll2003



Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Data preprocessing and label alignment complete!


In [8]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

# --- TASK 3: Model Setup ---
label_list = pos_labels
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

# --- TASK 4: Training ---
args = TrainingArguments(
    output_dir="./bert-pos-tagger",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, # Lowered for local laptop memory
    per_device_eval_batch_size=8,
    num_train_epochs=1,            # Set to 1 for faster local execution
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Starting training... (This may take a while on a local CPU)")
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Starting training... (This may take a while on a local CPU)


Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1756, training_loss=0.42812794922153063, metrics={'train_runtime': 7134.7016, 'train_samples_per_second': 1.968, 'train_steps_per_second': 0.246, 'total_flos': 297466474973688.0, 'train_loss': 0.42812794922153063, 'epoch': 1.0})

In [9]:
import numpy as np
import evaluate

# --- TASK 5: Evaluation ---
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[int(p)] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[int(l)] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

trainer.compute_metrics = compute_metrics
results = trainer.evaluate()

print("\n" + "="*30)
print("--- Final Evaluation Results ---")
print(f"Precision: {results['eval_precision']:.4f}")
print(f"Recall:    {results['eval_recall']:.4f}")
print(f"F1 Score:  {results['eval_f1']:.4f}")
print("="*30)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni


--- Final Evaluation Results ---
Precision: 0.9133
Recall:    0.9091
F1 Score:  0.9112


In [10]:
import torch

# --- TASK 6: Inference ---
text = "John works at Google in California"
inputs = tokenizer(text, return_tensors="pt")

# Move inputs to the same device as the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(device) for k, v in inputs.items()}
model.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

predictions = torch.argmax(logits, dim=2)
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

print(f"Input: {text}")
print("Output Predictions:")
for word, tag in zip(tokenizer.tokenize(text), predicted_token_class[1:-1]): # Skip [CLS] and [SEP]
    print(f"{word}: {tag}")

Input: John works at Google in California
Output Predictions:
john: NNP
works: VBZ
at: IN
google: NNP
in: IN
california: NNP


## Task 7: Comparison
* **POS Tagging:** This is grammar-level tagging (Easy). It assigns grammatical categories like noun, verb, or adjective to individual words.
* **Chunking:** This is phrase-level grouping (Medium). It groups adjacent tokens into larger constituent phrases, like Noun Phrases (NP) or Verb Phrases (VP), requiring the model to understand the boundaries between multi-word concepts.

## Task 8: Report / Blog
* **Challenges Faced:** The primary challenge was properly handling subword tokenization. Because BERT splits words into smaller pieces (e.g., separating suffixes), the original labels no longer matched the token count. Implementing a custom alignment function and using the `-100` index to ignore subwords during loss calculation was critical. Additionally, managing computational resources during training required adjusting batch sizes.
* **Observations:** Transformer models like BERT adapt incredibly well to sequence labeling tasks when provided with the correct token classification head. The `seqeval` metric is essential for these tasks, as standard accuracy is misleading due to the high volume of easy-to-predict padding or common tokens.